# Workshop 4 — Devnagari Script Recognition (FCN)
**Author:** Saharsh Pathak | **ID:** 2417371

Fully Connected Network for recognizing Devnagari handwritten characters.
Dataset: Devnagari Character Dataset (46 classes, 92,000 images)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Data Loading & Preprocessing

In [ ]:
# Data generators with augmentation
# NOTE: Update 'train_path' and 'test_path' to your dataset location
# Dataset: https://www.kaggle.com/datasets/rishianand/devanagari-character-set

IMG_SIZE = 32
BATCH_SIZE = 64
NUM_CLASSES = 46

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    validation_split=0.2
)

# Uncomment when dataset is available:
# train_gen = train_datagen.flow_from_directory(
#     'data/devnagari/train',
#     target_size=(IMG_SIZE, IMG_SIZE),
#     color_mode='grayscale',
#     batch_size=BATCH_SIZE,
#     class_mode='categorical',
#     subset='training'
# )

print(f'Image size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Number of classes: {NUM_CLASSES}')

## 2. FCN Model Architecture

In [ ]:
def build_fcn_model(input_shape=(32, 32, 1), num_classes=46):
    """
    Fully Connected Network (FCN) for Devnagari character classification.
    Architecture: Flatten -> Dense(512) -> BN -> Dropout -> Dense(256) -> BN -> Dropout -> Output
    """
    model = keras.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        layers.Flatten(),

        # Hidden layer 1
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        # Hidden layer 2
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Hidden layer 3
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),

        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ], name='Devnagari_FCN')

    return model

model = build_fcn_model()
model.summary()

## 3. Compile & Train

In [ ]:
# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint('best_fcn_model.h5', save_best_only=True, monitor='val_accuracy')
]

print('Model compiled successfully')
print('Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint')

# Training (uncomment when data is available):
# history = model.fit(
#     train_gen,
#     validation_data=val_gen,
#     epochs=50,
#     callbacks=callbacks
# )

## 4. Simulate Training Results

In [ ]:
# Simulated training history for demonstration
np.random.seed(42)
epochs = 30
train_acc = np.linspace(0.3, 0.88, epochs) + np.random.normal(0, 0.02, epochs)
val_acc = np.linspace(0.25, 0.84, epochs) + np.random.normal(0, 0.025, epochs)
train_loss = np.linspace(2.5, 0.4, epochs) + np.random.normal(0, 0.05, epochs)
val_loss = np.linspace(2.8, 0.55, epochs) + np.random.normal(0, 0.06, epochs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_acc, label='Train Accuracy', color='#0EA5E9', linewidth=2)
axes[0].plot(val_acc, label='Val Accuracy', color='#6366F1', linewidth=2, linestyle='--')
axes[0].set_title('FCN — Training vs Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_loss, label='Train Loss', color='#EF4444', linewidth=2)
axes[1].plot(val_loss, label='Val Loss', color='#F59E0B', linewidth=2, linestyle='--')
axes[1].set_title('FCN — Training vs Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Devnagari FCN Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Final Train Accuracy: {train_acc[-1]:.4f}')
print(f'Final Val Accuracy:   {val_acc[-1]:.4f}')